In [1]:
import pandas as pd 
dataset_manifest=pd.read_csv("../data/raw/metadata/metadata/dataset_manifest.csv")

In [2]:
dataset_manifest.head(30)
print(dataset_manifest.columns)

Index(['image_id', 'image_path', 'canonical_class', 'class_id', 'crop',
       'source_dataset', 'width', 'height', 'original_filename', 'mask_path'],
      dtype='str')


In [3]:
dataset_manifest.shape

(66953, 10)

In [4]:
import pandas as pd

train_split = pd.read_csv("../data/raw/outputs/outputs/train_split.csv")
test_split = pd.read_csv("../data/raw/outputs/outputs/test_split.csv")
val_split=pd.read_csv("../data/raw/outputs/outputs/val_split.csv")

print("train_split.csv:")
print(train_split.head())
print(train_split.columns.tolist())
print(train_split.shape)
print()

print("test_split.csv:")
print(test_split.head())
print(test_split.columns.tolist())
print(test_split.shape)

print("val_split.csv:")
print(val_split.head())
print(val_split.columns.tolist())
print(val_split.shape)

train_split.csv:
       image_id               image_path               canonical_class  \
0  img_00025856  images/img_00025856.JPG         Tomato - Spider Mites   
1  img_00038397  images/img_00038397.jpg  Garlic - garlic leaf blight    
2  img_00016302  images/img_00016302.JPG  Bell Pepper - Bacterial Spot   
3  img_00052992  images/img_00052992.jpg             Banana - Sigatoka   
4  img_00053059  images/img_00053059.jpg              Banana - Healthy   

   class_id         crop                    source_dataset  width  height  \
0       228       Tomato  PlantVillageDataset/PlantVillage    256     256   
1       121       Garlic                          plantseg    640     464   
2        40  Bell Pepper  PlantVillageDataset/PlantVillage    256     256   
3        18       Banana                   Multicrop/train    640     640   
4        13       Banana                   Multicrop/train    640     640   

                                   original_filename  mask_path  
0  423380

In [5]:
from pathlib import Path

train_classes = sorted([p.name for p in Path("../data/processed/train").iterdir() if p.is_dir()])
test_classes = sorted([p.name for p in Path("../data/processed/test").iterdir() if p.is_dir()])
val_classes=sorted([p.name for p in Path("../data/processed/val").iterdir() if p.is_dir()])

print(f"Train classes: {len(train_classes)}")
print(f"Test classes: {len(test_classes)}")
print(f"Match: {train_classes == test_classes==val_classes}")

Train classes: 211
Test classes: 211
Match: True


In [6]:
import sys
print(sys.executable)

e:\Sprayer\plant-disease-backend\.venv\Scripts\python.exe


In [7]:
import torch
import torchvision


In [8]:
from torchvision import transforms,datasets

In [9]:
from torch.utils.data import DataLoader,random_split



In [10]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [11]:
import torch
import torch.nn as nn
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights

# 1. Load EfficientNet-B0 with default pre-trained ImageNet weights
weights = EfficientNet_B0_Weights.DEFAULT
model = efficientnet_b0(weights=weights)

# 2. Inspect the classifier structure to see the input features of the final layer
# EfficientNet's classifier is typically a Sequential block ending with a Linear layer
in_features = model.classifier[1].in_features

# 3. Replace the final linear layer to output your exact number of classes (e.g., 5 classes)
num_classes = 5 
model.classifier[1] = nn.Linear(in_features, num_classes)

# 4. (Optional) Freeze the feature extractor backbone so it only trains the new classification head
for param in model.features.parameters():
    param.requires_grad = False

# Now only model.classifier parameters will be updated during the initial training phase

In [12]:
transform=weights.transforms()

In [13]:
train_data=datasets.ImageFolder(root="../data/processed/train",transform=transform)
test_data=datasets.ImageFolder(root="../data/processed/test",transform=transform)
val_data=datasets.ImageFolder(root="../data/processed/val",transform=transform)
train_data[0]

(tensor([[[ 1.0844,  1.1015,  1.1187,  ...,  0.3823,  0.3823,  0.3994],
          [ 1.0844,  1.1187,  1.1187,  ...,  0.4166,  0.4166,  0.4337],
          [ 1.0844,  1.1187,  1.1187,  ...,  0.4166,  0.4166,  0.4337],
          ...,
          [ 0.0398,  0.0227, -0.0116,  ..., -1.1589, -1.1589, -1.1418],
          [ 0.0227,  0.0227, -0.0287,  ..., -1.1589, -1.1418, -1.1247],
          [ 0.0056,  0.0056, -0.0458,  ..., -1.1247, -1.1075, -1.0733]],
 
         [[ 0.7829,  0.8004,  0.8179,  ...,  1.4657,  1.4657,  1.4657],
          [ 0.7829,  0.8179,  0.8179,  ...,  1.5007,  1.5007,  1.4657],
          [ 0.7829,  0.8179,  0.8179,  ...,  1.5007,  1.5007,  1.4657],
          ...,
          [ 0.2402,  0.2052,  0.1702,  ...,  0.1702,  0.1702,  0.2052],
          [ 0.2052,  0.2052,  0.1527,  ...,  0.2052,  0.2227,  0.2227],
          [ 0.1877,  0.1877,  0.1352,  ...,  0.2752,  0.2927,  0.2752]],
 
         [[ 0.7751,  0.7925,  0.8099,  ...,  1.6814,  1.6814,  1.6988],
          [ 0.7751,  0.8099,

In [14]:
len(val_data)


11012

In [15]:
train_loader = DataLoader(train_data, batch_size=16, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_data, batch_size=16, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_data, batch_size=16, shuffle=False, num_workers=2, pin_memory=True)

In [16]:
print(torch.cuda.is_available())

True


In [17]:
print(torch.cuda.memory_allocated())

0


In [18]:
device="cuda" if torch.cuda.is_available() else "cpu"
# 2. Move a dummy tensor to the GPU
x = torch.randn(1000, 1000, device=device)

# 3. Check memory allocated now (in bytes)
print(f"Memory allocated: {torch.cuda.memory_allocated() / 1024**2:.2f} MB")

Memory allocated: 3.81 MB


In [19]:
from torch import nn
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
weights = EfficientNet_B0_Weights.DEFAULT
model = efficientnet_b0(weights=weights)
for param in model.features.parameters():
    param.requires_grad =="False"
num_classes=len(train_classes)
in_features=model.classifier[1].in_features
model.classifier = nn.Sequential(
    nn.Dropout(p=0.2, inplace=True),
    nn.Linear(in_features=in_features, out_features=num_classes)
)
model = model.to(device)
print(f"VRAM after loading model: {torch.cuda.memory_allocated() / 1024**2:.2f} MB")

VRAM after loading model: 20.37 MB


In [20]:
images,labels=next(iter(train_loader))
images,labels=images.to(device),labels.to(device)
outputs = model(images)
print(f"Output shape: {outputs.shape}")  # Should be [16, num_classes]
print(f"VRAM after 1 batch forward pass: {torch.cuda.memory_allocated() / 1024**2:.2f} MB")

Output shape: torch.Size([16, 211])
VRAM after 1 batch forward pass: 1393.32 MB


In [21]:
import torch
import torch.nn as nn
import torch.optim as optim

def run_smoke_test(model, train_loader, loss_fn, optimizer, device, iterations: int = 20):
    """
    Overfits a single batch to verify that gradients flow and the pipeline functions correctly.
    """
    model.train()
    
    # Extract exactly one batch from the dataloader
    images, labels = next(iter(train_loader))
    images, labels = images.to(device), labels.to(device)
    
    print(f"Starting smoke test on 1 batch ({images.size(0)} images) for {iterations} iterations...")
    
    for i in range(iterations):
        optimizer.zero_grad()
        
        outputs = model(images)
        loss = loss_fn(outputs, labels)
        
        loss.backward()
        optimizer.step()
        
        _, preds = outputs.max(1)
        accuracy = (preds == labels).float().mean().item() * 100
        
        print(f"Step {i+1:02d}/{iterations} | Loss: {loss.item():.4f} | Batch Accuracy: {accuracy:.1f}%")


loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-3)
run_smoke_test(model, train_loader, loss_fn, optimizer, device)

Starting smoke test on 1 batch (16 images) for 20 iterations...


KeyboardInterrupt: 